## Dependencias

In [8]:
%pip install -qU langchain-community langchain-core langchain-text-splitters \
    langchain-huggingface pypdf chromadb langgraph pandas sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


## Dataset: Netflix Titles

In [1]:
import pandas as pd
import io
from google.colab import files

print("Selecciona el archivo CSV de tu dataset desde tu computadora:")
uploaded = files.upload()

# Obtener automáticamente el nombre del archivo cargado
csv_filename = list(uploaded.keys())[0]

# Cargar el DataFrame directamente desde el contenido en memoria
df_netflix = pd.read_csv(io.BytesIO(uploaded[csv_filename]))

print(f"\nArchivo '{csv_filename}' cargado con éxito.")
print(f"Filas: {df_netflix.shape[0]} | Columnas: {df_netflix.shape[1]}")
print("Columnas disponibles:", df_netflix.columns.tolist())
df_netflix.head(2)

Selecciona el archivo CSV de tu dataset desde tu computadora:


Saving netflix_titles.csv to netflix_titles.csv

Archivo 'netflix_titles.csv' cargado con éxito.
Filas: 8807 | Columnas: 12
Columnas disponibles: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."


### Limpieza del dataset

Para preparar el dataset de Netflix se reemplazaron los datos vacíos en columnas como el director o el elenco por espacios en blanco, evitando problemas y errores al juntar los textos más adelante.

Luego se hizo una limpieza simple para quitar espacios de más y saltos de línea repetidos, sin tocar el vocabulario ni recortar palabras. La idea es dejar el texto lo más natural y fluido posible, ya que los modelos de lenguaje modernos entienden mejor las oraciones completas y sus relaciones cuando conservan su orden habitual.

Con los textos limpios, se unieron las partes más importantes de cada película —como título, género y descripción— anteponiendo etiquetas sencillas para que el sistema distinga claramente qué representa cada dato dentro de una sola frase.

Finalmente, se separó lo esencial de lo accesorio: en el texto principal se dejaron solo los datos que definen la trama y el argumento para no sobrecargar la búsqueda conceptual, mientras que otros datos como el país o el año se guardaron como etiquetas de consulta. De este modo, la máquina puede buscar por afinidad temática sin confundirse y, a la vez, consultar o filtrar directamente por esos datos específicos cuando sea necesario.

In [2]:
import re

# 1. Definir columnas relevantes de texto
text_cols = ['title', 'type', 'director', 'cast', 'listed_in', 'description']

# Rellenar valores nulos con cadenas vacías para evitar errores de concatenación
for col in text_cols:
    if col in df_netflix.columns:
        df_netflix[col] = df_netflix[col].fillna('')

# 2. Función de normalización básica (preserva la semántica original)
def clean_text(s):
    if not s or not str(s).strip():
        return ""
    # Reemplaza secuencias de múltiples espacios o saltos de línea por un único espacio
    return re.sub(r'\s+', ' ', str(s)).strip()

for col in text_cols:
    if col in df_netflix.columns:
        df_netflix[col] = df_netflix[col].apply(clean_text)

# 3. Concatenación estructurada con prefijos semánticos
# Esto ayuda al modelo de embeddings (ej. MiniLM) a distinguir qué es título, género, trama, etc.
df_netflix['text_for_embedding'] = (
    "Title: " + df_netflix['title'] + ". " +
    "Type: " + df_netflix['type'] + ". " +
    "Genres: " + df_netflix['listed_in'] + ". " +
    "Director: " + df_netflix['director'] + ". " +
    "Cast: " + df_netflix['cast'] + ". " +
    "Description: " + df_netflix['description']
).str.strip()

# 4. Verificación rápida del resultado
print("Muestra de texto preparado para embeddings:")
print("-" * 60)
print(df_netflix['text_for_embedding'].iloc[0])
print("-" * 60)
df_netflix[['title', 'text_for_embedding']].head(3)

Muestra de texto preparado para embeddings:
------------------------------------------------------------
Title: Dick Johnson Is Dead. Type: Movie. Genres: Documentaries. Director: Kirsten Johnson. Cast: . Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.
------------------------------------------------------------


,title,text_for_embedding
0,Dick Johnson Is Dead,Title: Dick Johnson Is Dead. Type: Movie. Genr...
1,Blood & Water,Title: Blood & Water. Type: TV Show. Genres: I...
2,Ganglands,Title: Ganglands. Type: TV Show. Genres: Crime...


In [3]:
from langchain_core.documents import Document

# Limpieza previa de nulos
df_netflix['country'] = df_netflix['country'].fillna('Unknown')
df_netflix['release_year'] = df_netflix['release_year'].fillna(0)

netflix_documents = []
for idx, row in df_netflix.iterrows():
    doc = Document(
        page_content=row['text_for_embedding'],  # Lo que busca semánticamente
        metadata={
            "show_id": str(row.get('show_id', idx)),
            "title": str(row['title']),
            "type": str(row['type']),
            "country": str(row['country']),
            "release_year": int(row['release_year']),
            "source_type": "kaggle_netflix"
        }
    )
    netflix_documents.append(doc)

### Indexación en Pinecone

Se usa All Mini, es un modelo entrenado casi exclusivamente en inglés, por lo que funciona correctamente con datos y querys en inglés, no tanto en español.

In [5]:
%pip install -qU pinecone-client langchain-pinecone

In [9]:
import os
import time
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

pinecone_api_key = userdata.get("pinecone") #agreguen con or como lo tengan en los secretos de Colab
if not pinecone_api_key:
    raise ValueError("No se encontró la API Key de Pinecone en los Secretos de Colab.")

os.environ["PINECONE_API_KEY"] = pinecone_api_key
pc = Pinecone(api_key=pinecone_api_key)

index_netflix_name = "netflix-catalog"
dimension = 384
metric = "cosine"
spec = ServerlessSpec(cloud="aws", region="us-east-1")

existing_indexes = [idx.name for idx in pc.list_indexes()]

if index_netflix_name not in existing_indexes:
    print(f"Creando índice '{index_netflix_name}' en Pinecone...")
    pc.create_index(
        name=index_netflix_name,
        dimension=dimension,
        metric=metric,
        spec=spec
    )
    while not pc.describe_index(index_netflix_name).status["ready"]:
        time.sleep(1)
    print(f"Índice '{index_netflix_name}' listo.")
else:
    print(f"El índice '{index_netflix_name}' ya existe. Conectando...")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print(f"Indexando {len(netflix_documents)} títulos en Pinecone...")
vectorstore_netflix = PineconeVectorStore.from_documents(
    documents=netflix_documents,
    embedding=embeddings,
    index_name=index_netflix_name
)

print(f"¡Catálogo de Netflix indexado con éxito en '{index_netflix_name}'!")

Creando índice 'netflix-catalog' en Pinecone...
Índice 'netflix-catalog' listo.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexando 8807 títulos en Pinecone...
¡Catálogo de Netflix indexado con éxito en 'netflix-catalog'!


## Probando con retriever

In [10]:
# 1. Instanciar el retriever a partir del vectorstore de Netflix
retriever_netflix = vectorstore_netflix.as_retriever(search_kwargs={"k": 3})

print("=" * 65)
print("TEST 1: BÚSQUEDA SEMÁNTICA PURA (Concepto / Trama)")
print("=" * 65)

# Búsqueda temática (en inglés debido a all-MiniLM-L6-v2)
query_1 = "detective solving murder mystery in a small dark town"
docs_1 = retriever_netflix.invoke(query_1)

for i, doc in enumerate(docs_1, start=1):
    meta = doc.metadata
    print(f"\nResultado {i}:")
    print(f"  • Título:       {meta.get('title')} ({meta.get('type')})")
    print(f"  • País:         {meta.get('country')} | Año: {meta.get('release_year')}")
    print(f"  • Fragmento:    {doc.page_content[:150]}...")

print("\n" + "=" * 65)
print("TEST 2: BÚSQUEDA CONCEPTUAL DISTINTA (Ciencia Ficción)")
print("=" * 65)

query_2 = "space travel and exploration of alien civilizations"
docs_2 = retriever_netflix.invoke(query_2)

for i, doc in enumerate(docs_2, start=1):
    meta = doc.metadata
    print(f"\nResultado {i}:")
    print(f"  • Título:       {meta.get('title')} ({meta.get('type')})")
    print(f"  • País:         {meta.get('country')} | Año: {meta.get('release_year')}")
    print(f"  • Fragmento:    {doc.page_content[:150]}...")

print("\n" + "=" * 65)
print("TEST 3: FILTRADO DETERMINISTA POR METADATOS (Solo Películas de España)")
print("=" * 65)

# Búsqueda semántica combinada con filtro en Pinecone
docs_3 = vectorstore_netflix.similarity_search(
    "crime and thrilling heist",
    k=3,
    filter={"country": {"$eq": "Spain"}, "type": {"$eq": "Movie"}}
)

if not docs_3:
    print("No se encontraron registros que cumplan estrictamente el filtro.")
else:
    for i, doc in enumerate(docs_3, start=1):
        meta = doc.metadata
        print(f"\nResultado {i}:")
        print(f"  • Título:       {meta.get('title')}")
        print(f"  • País:         {meta.get('country')} | Tipo: {meta.get('type')}")
        print(f"  • Fragmento:    {doc.page_content[:150]}...")

TEST 1: BÚSQUEDA SEMÁNTICA PURA (Concepto / Trama)

Resultado 1:
  • Título:       Dark Crimes (Movie)
  • País:         United Kingdom, Poland, United States | Año: 2016.0
  • Fragmento:    Title: Dark Crimes. Type: Movie. Genres: Dramas, Thrillers. Director: Alexandros Avranas. Cast: Jim Carrey, Marton Csokas, Charlotte Gainsbourg, Agata...

Resultado 2:
  • Título:       Small Town Crime (Movie)
  • País:         United States | Año: 2017.0
  • Fragmento:    Title: Small Town Crime. Type: Movie. Genres: Thrillers. Director: Eshom Nelms, Ian Nelms. Cast: John Hawkes, Anthony Anderson, Octavia Spencer, Rober...

Resultado 3:
  • Título:       The Murder Detectives (TV Show)
  • País:         United Kingdom | Año: 2015.0
  • Fragmento:    Title: The Murder Detectives. Type: TV Show. Genres: British TV Shows, Crime TV Shows, Docuseries. Director: . Cast: . Description: This series tracks...

TEST 2: BÚSQUEDA CONCEPTUAL DISTINTA (Ciencia Ficción)

Resultado 1:
  • Título:       The Searc

-------------------------------------------------------------

## Papers

### Subida de papers

In [ ]:
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

print("Selecciona los 6 archivos PDF de los papers desde tu computadora (puedes seleccionarlos todos a la vez):")
uploaded_papers = files.upload()

# Lista de documentos sin procesar
raw_papers_docs = []

# Procesar cada PDF subido
for filename in uploaded_papers.keys():
    if filename.lower().endswith(".pdf"):
        # files.upload() ya guardó el archivo en el directorio actual (/content)
        loader = PyPDFLoader(filename)
        pages = loader.load()

        # Enriquecer la metadata con el nombre del paper
        for page in pages:
            page.metadata["file_name"] = filename
            page.metadata["source_type"] = "research_paper"

        raw_papers_docs.extend(pages)
        print(f"Cargado: {filename} ({len(pages)} páginas)")

print(f"\nTotal de papers cargados: {len(uploaded_papers)}")
print(f"Total de páginas procesadas: {len(raw_papers_docs)}")

if raw_papers_docs:
    print(f"\nMuestra de la primera página de '{raw_papers_docs[0].metadata['file_name']}':")
    print(raw_papers_docs[0].page_content[:300] + "...")